# AQI Watch — Ortigas Station (Mandaluyong)
### Data Understanding & Data Preparation notebook (CRISP-DM)

| Part | What it does |
|---|---|
| **0. Setup** | Settings, thresholds, Google Drive, helper functions |
| **1. Data retrieval** | Builds the 4 source tables separately (PM2.5, Weather, Urban environment, Time features) |
| **2. EDA — Data quality** | Checks each table on 6 dimensions: uniqueness, completeness, validity, consistency, accuracy, timeliness |
| **3. Outliers** | Finds and labels outliers; removes only impossible values |
| **4. Merging** | Joins all 4 tables into one hourly table, fills short gaps, builds the DENR risk label |
| **5. EDA — Merged table** | Class balance, daily pattern, weather relationships |
| **6. Decision summary** | Every decision made in this notebook with its justification |

**How to run**
1. Add your OpenAQ key: 🔑 *Secrets* (left sidebar) → name `OPENAQ_API_KEY` → turn on *Notebook access*.
   Groupmates without a key can set `USE_SAVED_TABLES = True` to load the CSVs already saved in the shared Drive folder.
2. *Runtime → Run all*. Allow Google Drive access when asked.
3. Cells marked **📝 Fill in after running** need the group to write down what the results show (see the documentation guide).

---
# Part 0 — Setup

In [ ]:
!pip -q install holidays

import os, json, time, warnings
from datetime import datetime, timezone
import requests
import numpy as np
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", message="Converting to PeriodArray")   # harmless: monthly grouping drops the time zone label

### Settings and thresholds
Every threshold used later is set here, in one place, so the group can change it and re-run. The reason for each value is in the comment and is explained again where it is used.

In [ ]:
# ---------- Study settings ----------
STATION_LAT, STATION_LON = 14.5866, 121.0566   # Ortigas Center (approx.); replaced by the station's exact coordinates in Table 1a
SEARCH_RADIUS_M = 5000                          # how far to look for OpenAQ PM2.5 stations
ORTIGAS_LOCATION_ID = 3015285                   # OpenAQ location of the Ortigas station (Clarity low-cost sensor)
START_DATE = "2024-11-01"
END_DATE   = "2026-10-31"
LOCAL_TZ   = "Asia/Manila"

# ---------- Data source switch ----------
USE_SAVED_TABLES = False   # True = read the CSVs in DATA_DIR instead of calling the APIs (no API key needed)
DRIVE_FOLDER = "/content/drive/MyDrive/AQI_Watch/data"   # shared group folder (see the documentation guide)

# ---------- Data-quality thresholds ----------
MIN_HOUR_COVERAGE = 75   # % of expected raw readings an hourly average needs; 75% is the usual data-capture rule in air quality reporting
MONTH_USABLE_PCT  = 75   # a month counts as usable if ≥75% of its hours have PM2.5
FLATLINE_HOURS    = 6    # the same PM2.5 value 6+ hours in a row = stuck sensor
PM25_MAX_VALID    = 1000 # µg/m³; above this is a sensor error, not air (even severe haze episodes stay well below)
MAX_INTERP_GAP_H  = 3    # only gaps of ≤3 hours are interpolated (longer gaps would flatten the rush-hour peaks)
ROLLING_MIN_HOURS = 18   # a 24-h average needs ≥18 valid hours (75% of 24)

# ---------- DENR AQI breakpoints for PM2.5, 24-h average, µg/m³ (DAO 2020-14) ----------
DENR_BINS   = [-np.inf, 25, 35, 45, 55, 90, np.inf]
DENR_LABELS = ["Good", "Fair", "Unhealthy", "Very Unhealthy", "Acutely Unhealthy", "Emergency"]
# DAO 2020-14 names: Good 0–25, Fair 25.1–35, Unhealthy for Sensitive Groups 35.1–45,
# Very Unhealthy 45.1–55, Acutely Unhealthy 55.1–90, Emergency ≥91 (values between 90 and 91 are put in Emergency)

In [ ]:
# Google Drive: the group's shared data folder. Falls back to a local folder outside Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_FOLDER
except Exception:
    DATA_DIR = "data"
FIG_DIR = os.path.join(os.path.dirname(os.path.abspath(DATA_DIR)), "figures")   # charts for the report go here
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print("Tables are saved to / loaded from:", DATA_DIR)
print("Save report figures to:", FIG_DIR)

META_PATH = f"{DATA_DIR}/run_metadata.json"
def load_meta():
    return json.load(open(META_PATH)) if os.path.exists(META_PATH) else {}
def update_meta(key, value):
    meta = load_meta(); meta[key] = value
    json.dump(meta, open(META_PATH, "w"), indent=2, default=str)
def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

In [ ]:
# ---------- Helper functions used throughout ----------
def to_local(s):
    """Parse any timestamp column into Asia/Manila time with one consistent dtype (needed for joins)."""
    return pd.to_datetime(s, utc=True).dt.tz_convert(LOCAL_TZ).astype(f"datetime64[ns, {LOCAL_TZ}]")

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def true_runs(mask):
    """Start, end and length of every run of consecutive True values in a time-indexed boolean Series."""
    mask = mask.astype(bool)
    run_id = mask.ne(mask.shift()).cumsum()
    df = pd.DataFrame({"m": mask.values, "run": run_id.values, "t": mask.index})
    runs = df[df["m"]].groupby("run").agg(start=("t", "first"), end=("t", "last"), hours=("m", "size"))
    return runs.reset_index(drop=True)

def hourly_index():
    return pd.date_range(f"{START_DATE} 00:00", f"{END_DATE} 23:00", freq="h", tz=LOCAL_TZ)

# Data-quality log: every check in Part 2 adds one row here; the scorecard at the end of Part 2 is built from it
dq_rows = []
def dq(dimension, table, check, result, ok):
    dq_rows.append({"dimension": dimension, "table": table, "check": check,
                    "result": result, "status": "OK" if ok else "FLAG"})
    print(f"[{'OK  ' if ok else 'FLAG'}] {dimension:<12} | {table:<10} | {check}: {result}")

# ---------- Chart style ----------
BLUE, ORANGE, GRAY, INK = "#2a78d6", "#eb6834", "#8a8985", "#52514e"
DENR_COLORS = ["#3a9d23", "#e8b400", "#f08a24", "#d7301f", "#7b3294", "#7e0023"]   # green → maroon, as on DENR advisories
SEQ_CMAP = LinearSegmentedColormap.from_list("seq_blue", ["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
DIV_CMAP = LinearSegmentedColormap.from_list("div", ["#1c5cab", "#86b6ef", "#f0efec", "#ec8a89", "#b52a2a"])
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
    "axes.edgecolor": "#b5b4ae", "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.titlesize": 12, "axes.axisbelow": True, "axes.titleweight": "bold", "lines.linewidth": 1.5,
})

---
# Part 1 — Data retrieval (one table per source)

| Table | Source | Holds | Granularity |
|---|---|---|---|
| 1a | OpenAQ v3 | PM2.5 stations near Ortigas (lookup) | one row per sensor |
| 1 | OpenAQ v3 | Hourly PM2.5 at the Ortigas station | hourly, has gaps, UTC |
| 1b | OpenAQ v3 | Hourly PM2.5 at the nearest *other* station (used only to check accuracy) | hourly |
| 2 | Open-Meteo archive | Temperature, humidity, wind, rainfall | hourly, complete, Asia/Manila |
| 3 | OpenStreetMap (Overpass) | Industrial areas, roads, fuel stations, bus stops within 1 km | one static row |
| 4 | Generated | Hour, weekday, weekend, rush-hour period, PH holidays, season | hourly, complete |

Part 1 only **retrieves** data. Nothing is cleaned or dropped here, so Part 2 can measure the quality of the data exactly as it arrived. Every table is saved to the shared Drive folder together with the time it was retrieved (used for the timeliness check).

In [ ]:
# OpenAQ API key (not needed when loading saved tables)
OPENAQ_API_KEY = None
try:
    from google.colab import userdata
    OPENAQ_API_KEY = userdata.get("OPENAQ_API_KEY")
except Exception:
    if not USE_SAVED_TABLES:
        from getpass import getpass
        OPENAQ_API_KEY = getpass("Paste your OpenAQ API key: ")

OPENAQ_BASE = "https://api.openaq.org/v3"

def openaq_get(path, params=None, retries=4):
    """GET from OpenAQ v3, waiting and retrying when the rate limit (60 requests/min) is hit."""
    for _ in range(retries):
        r = requests.get(f"{OPENAQ_BASE}{path}", headers={"X-API-Key": OPENAQ_API_KEY}, params=params, timeout=60)
        if r.status_code == 429:
            wait = int(r.headers.get("x-ratelimit-reset", 60) or 60)
            print(f"Rate limited, waiting {wait}s ..."); time.sleep(wait); continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"OpenAQ request kept failing: {path}")

def fetch_sensor_hours(sensor_id, date_from, date_to, page_size=1000):
    """Hourly averages for one sensor. Raw: no de-duplication or cleaning."""
    out, page = [], 1
    while True:
        js = openaq_get(f"/sensors/{sensor_id}/hours", {
            "datetime_from": f"{date_from}T00:00:00Z", "datetime_to": f"{date_to}T23:59:59Z",
            "limit": page_size, "page": page})
        results = js.get("results", [])
        if not results:
            break
        for r in results:
            summary, coverage = r.get("summary") or {}, r.get("coverage") or {}
            out.append({
                "datetime_utc":     r["period"]["datetimeFrom"]["utc"],   # start of the hour
                "pm25":             r["value"],
                "pm25_min":         summary.get("min"),
                "pm25_max":         summary.get("max"),
                "observed_count":   coverage.get("observedCount"),
                "percent_coverage": coverage.get("percentComplete"),
            })
        print(f"  page {page}: {len(results)} rows")
        if len(results) < page_size:
            break
        page += 1
        time.sleep(1.1)
    return pd.DataFrame(out)

## Table 1a — PM2.5 stations near Ortigas

In [ ]:
path = f"{DATA_DIR}/table1a_openaq_stations.csv"
if USE_SAVED_TABLES and os.path.exists(path):
    stations = pd.read_csv(path)
else:
    resp = openaq_get("/locations", {"coordinates": f"{STATION_LAT},{STATION_LON}",
                                     "radius": SEARCH_RADIUS_M, "parameters_id": 2, "limit": 100})
    rows = []
    for loc in resp["results"]:
        for s in loc.get("sensors", []):
            if s["parameter"]["name"] == "pm25":
                rows.append({
                    "location_id": loc["id"], "location_name": loc["name"], "locality": loc.get("locality"),
                    "provider": (loc.get("provider") or {}).get("name"),
                    "lat": loc["coordinates"]["latitude"], "lon": loc["coordinates"]["longitude"],
                    "sensor_id": s["id"], "units": s["parameter"]["units"],
                    "first_reading_utc": (loc.get("datetimeFirst") or {}).get("utc"),
                    "last_reading_utc":  (loc.get("datetimeLast") or {}).get("utc"),
                })
    stations = pd.DataFrame(rows)
    stations.to_csv(path, index=False)
    update_meta("table1a_retrieved_utc", now_utc())

stations["distance_km"] = haversine_km(STATION_LAT, STATION_LON, stations["lat"], stations["lon"]).round(2)
stations.sort_values("distance_km")

In [ ]:
# Pick the Ortigas sensor: OpenAQ location 3015285 (Clarity low-cost sensor), as stated in the project documentation.
# If that location isn't in the table above, the first location named "Ortigas" is used instead.
# To override, put the right sensor_id from the table above in SENSOR_ID.
SENSOR_ID = None

if SENSOR_ID is None:
    match = stations[stations["location_id"] == ORTIGAS_LOCATION_ID]
    if match.empty:
        print(f"Location {ORTIGAS_LOCATION_ID} not found within {SEARCH_RADIUS_M} m; falling back to the name 'Ortigas'.")
        match = stations[stations["location_name"].str.contains("ortigas", case=False, na=False)]
    if match.empty:
        raise ValueError("No Ortigas location found. Check the table above and set SENSOR_ID by hand.")
    SENSOR_ID = int(match.iloc[0]["sensor_id"])

station = stations.loc[stations["sensor_id"] == SENSOR_ID].iloc[0]
STATION_NAME = station["location_name"]
STATION_LAT, STATION_LON = float(station["lat"]), float(station["lon"])
update_meta("station", {"name": STATION_NAME, "sensor_id": SENSOR_ID, "lat": STATION_LAT, "lon": STATION_LON,
                        "provider": station["provider"]})
station

## Table 1 — Hourly PM2.5 (raw)

In [ ]:
path = f"{DATA_DIR}/table1_openaq_pm25_hourly_raw.csv"
if USE_SAVED_TABLES and os.path.exists(path):
    aq_raw = pd.read_csv(path)
else:
    aq_raw = fetch_sensor_hours(SENSOR_ID, START_DATE, END_DATE)
    aq_raw.to_csv(path, index=False)
    update_meta("table1_retrieved_utc", now_utc())

aq_raw["datetime_utc"]   = pd.to_datetime(aq_raw["datetime_utc"], utc=True)
aq_raw["datetime_local"] = to_local(aq_raw["datetime_utc"])
aq_raw = aq_raw.sort_values("datetime_utc").reset_index(drop=True)
aq_raw = aq_raw[["datetime_utc", "datetime_local", "pm25", "pm25_min", "pm25_max", "observed_count", "percent_coverage"]]
print(aq_raw.shape)
aq_raw.head(10)

## Table 1b — Nearest other PM2.5 station (accuracy cross-check only)
There is no "true" PM2.5 value to compare the Ortigas station against. The closest thing is a second station nearby: if both stations rise and fall together, the Ortigas readings are believable. This table is **not** merged into the final dataset.

In [ ]:
CROSS_CHECK_NEIGHBOR = True    # set False to skip (saves ~20 API calls)

neighbor = None
others = stations[stations["sensor_id"] != SENSOR_ID].sort_values("distance_km")
path = f"{DATA_DIR}/table1b_openaq_neighbor_hourly_raw.csv"
if CROSS_CHECK_NEIGHBOR and not others.empty:
    nb_info = others.iloc[0]
    if USE_SAVED_TABLES and os.path.exists(path):
        neighbor = pd.read_csv(path)
    else:
        print(f"Neighbor: {nb_info['location_name']} (sensor {nb_info['sensor_id']}, {nb_info['distance_km']} km away)")
        neighbor = fetch_sensor_hours(int(nb_info["sensor_id"]), START_DATE, END_DATE)
        neighbor.to_csv(path, index=False)
        update_meta("neighbor", {"name": nb_info["location_name"], "sensor_id": int(nb_info["sensor_id"]),
                                 "distance_km": float(nb_info["distance_km"]), "provider": nb_info["provider"]})
    neighbor["datetime_local"] = to_local(neighbor["datetime_utc"])
    print(neighbor.shape)
else:
    print("No neighbor station available; the accuracy cross-check will be skipped.")

## Table 2 — Weather (Open-Meteo historical archive)
`temp_c`, `humidity_pct` and wind are values **at** the timestamp; `precip_mm` and `rain_mm` are totals for the **hour before** the timestamp.

In [ ]:
path = f"{DATA_DIR}/table2_openmeteo_weather_hourly.csv"
if USE_SAVED_TABLES and os.path.exists(path):
    weather = pd.read_csv(path)
else:
    wx = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": STATION_LAT, "longitude": STATION_LON,
        "start_date": START_DATE, "end_date": END_DATE,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation,rain",
        "wind_speed_unit": "kmh", "timezone": LOCAL_TZ}, timeout=120)
    wx.raise_for_status()
    wx_json = wx.json()
    weather = pd.DataFrame(wx_json["hourly"]).rename(columns={
        "time": "datetime_local", "temperature_2m": "temp_c", "relative_humidity_2m": "humidity_pct",
        "wind_speed_10m": "wind_speed_kmh", "wind_direction_10m": "wind_dir_deg",
        "precipitation": "precip_mm", "rain": "rain_mm"})
    # Open-Meteo gives local times without an offset; the Philippines has no daylight saving, so this is safe
    weather["datetime_local"] = pd.to_datetime(weather["datetime_local"]).dt.tz_localize(LOCAL_TZ)
    weather.to_csv(path, index=False)
    update_meta("table2_retrieved_utc", now_utc())
    update_meta("weather_grid", {"lat": wx_json["latitude"], "lon": wx_json["longitude"]})

weather["datetime_local"] = to_local(weather["datetime_local"])
print(weather.shape)
weather.head(10)

## Table 3 — Urban environment (OpenStreetMap)
Counts of features within 1 km of the station. MMDA traffic volume is published in reports, not an API; add it by hand as a column if the group gets the figure.

In [ ]:
ENV_RADIUS_M = 1000
path = f"{DATA_DIR}/table3_osm_environment.csv"
if USE_SAVED_TABLES and os.path.exists(path):
    environment = pd.read_csv(path)
else:
    around = f"(around:{ENV_RADIUS_M},{STATION_LAT},{STATION_LON})"
    q = f"""
    [out:json][timeout:90];
    ( way["landuse"="industrial"]{around}; relation["landuse"="industrial"]{around}; ); out count;
    way["highway"~"^(motorway|trunk|primary)$"]{around}; out count;
    way["highway"~"^(secondary|tertiary)$"]{around}; out count;
    node["amenity"="fuel"]{around}; out count;
    node["highway"="bus_stop"]{around}; out count;
    """
    r = requests.post("https://overpass-api.de/api/interpreter", data={"data": q},
                      headers={"User-Agent": "AQI-Watch-student-project"}, timeout=120)
    r.raise_for_status()
    counts = [int(el["tags"]["total"]) for el in r.json()["elements"]]
    environment = pd.DataFrame([{
        "station_name": STATION_NAME, "lat": STATION_LAT, "lon": STATION_LON, "radius_m": ENV_RADIUS_M,
        "industrial_areas": counts[0], "major_road_segments": counts[1], "secondary_road_segments": counts[2],
        "fuel_stations": counts[3], "bus_stops": counts[4]}])
    environment.to_csv(path, index=False)
    update_meta("table3_retrieved_utc", now_utc())
environment

## Table 4 — Time features (generated)
Covers **every hour** of the study period, so it is also the backbone (spine) for the merge in Part 4.

In [ ]:
def day_period(h):
    if 6 <= h <= 9:   return "morning_rush"
    if 10 <= h <= 16: return "midday"
    if 17 <= h <= 20: return "evening_rush"
    return "night"

ph_holidays = holidays.Philippines(years=range(int(START_DATE[:4]), int(END_DATE[:4]) + 1))

time_tbl = pd.DataFrame({"datetime_local": hourly_index().astype(f"datetime64[ns, {LOCAL_TZ}]")})
t = time_tbl["datetime_local"].dt
time_tbl["date"]        = t.date
time_tbl["hour"]        = t.hour
time_tbl["day_of_week"] = t.dayofweek          # 0 = Monday
time_tbl["day_name"]    = t.day_name()
time_tbl["is_weekend"]  = (t.dayofweek >= 5).astype(int)
time_tbl["month"]       = t.month
time_tbl["day_period"]  = time_tbl["hour"].apply(day_period)
time_tbl["is_holiday"]  = time_tbl["date"].apply(lambda d: int(d in ph_holidays))
time_tbl["season"]      = np.where(t.month.isin([6, 7, 8, 9, 10, 11]), "wet", "dry")   # PAGASA-style split
time_tbl.to_csv(f"{DATA_DIR}/table4_time_features_hourly.csv", index=False)
print(time_tbl.shape)
time_tbl.head(10)

---
# Part 2 — EDA: Data quality

Each table is checked on six dimensions. Every check prints one line (`OK` or `FLAG`) and is collected into a **scorecard** at the end of this part.

| Dimension | Question it answers | Main checks |
|---|---|---|
| **Uniqueness** | Is every hour recorded once? | duplicate timestamps, duplicate rows |
| **Completeness** | How much is missing, and where? | % of hours present, monthly coverage, gap lengths, missing values per column |
| **Validity** | Are values inside allowed ranges and formats? | range rules per column (e.g. humidity 0–100) |
| **Consistency** | Do values agree with each other, within and across tables? | min ≤ avg ≤ max, rain ≤ precipitation, regular hourly steps, same time zone, same location |
| **Accuracy** | Are the values close to the real air? | stuck sensor, agreement with a nearby station, sensor type, weather vs known seasons |
| **Timeliness** | Is the data recent enough and up to date? | latest reading vs study end, station still reporting, retrieval date |

**Order of the checks.** Uniqueness goes first: duplicates would inflate every later count (an hour recorded twice looks like extra coverage). Then completeness, validity and consistency, which can be measured with rules. Accuracy and timeliness come last because they build on the earlier results.

**Rule for this part:** checks only *measure*. Anything that changes the data (dropping, setting to missing, interpolating) happens in Parts 3 and 4, so every change is visible and reversible.

In [ ]:
# Quick profile of every table: size, types, missing values
for name, df in {"Table 1 PM2.5": aq_raw, "Table 2 Weather": weather,
                 "Table 3 Environment": environment, "Table 4 Time": time_tbl}.items():
    print(f"\n=== {name}: {df.shape[0]:,} rows × {df.shape[1]} columns ===")
    print(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(),
                        "missing_%": (df.isna().mean() * 100).round(2)}).to_string())

## 2.1 Uniqueness
**Why it matters:** each hour must appear once. A duplicated hour gets double weight in training and breaks the one-to-one join in Part 4.
OpenAQ can return the same hour twice when a station re-uploads data, so Table 1 is the one most at risk.

In [ ]:
exact_dups    = aq_raw.duplicated().sum()
ts_dups       = aq_raw.duplicated("datetime_utc").sum()
conflict_dups = ts_dups - exact_dups
dq("Uniqueness", "Table 1", "exact duplicate rows", exact_dups, exact_dups == 0)
dq("Uniqueness", "Table 1", "same hour, different values", conflict_dups, conflict_dups == 0)
if conflict_dups:
    display(aq_raw[aq_raw.duplicated("datetime_utc", keep=False)].head(20))

d = weather.duplicated("datetime_local").sum()
dq("Uniqueness", "Table 2", "duplicate hours", d, d == 0)
d = time_tbl.duplicated("datetime_local").sum()
dq("Uniqueness", "Table 4", "duplicate hours", d, d == 0)
dq("Uniqueness", "Table 3", "rows (expected 1 station)", len(environment), len(environment) == 1)
d = stations.duplicated("sensor_id").sum()
dq("Uniqueness", "Table 1a", "duplicate sensor IDs", d, d == 0)

**Decision — resolving duplicates (applied in Part 3):**
- *Exact duplicates* → keep one copy. They carry no extra information.
- *Same hour, different values* → keep the row with the higher `observed_count` (built from more raw readings, so more reliable). If tied, keep the later row returned by the API (the latest upload usually corrects earlier ones).

## 2.2 Completeness
**Why it matters:** the proposal's own limitation is that station downtime creates gaps. The gap pattern decides (a) which months are usable, (b) how many gaps can be filled, and (c) how many rows survive into modeling.
Completeness is measured against the full hourly calendar (`START_DATE` to `END_DATE`), not against the rows that happen to exist.

In [ ]:
expected = hourly_index()
pm = (aq_raw.drop_duplicates("datetime_utc").set_index("datetime_local")["pm25"]
        .reindex(expected))                       # full hourly calendar; NaN = no reading
present_pct = pm.notna().mean() * 100
dq("Completeness", "Table 1", "hours with PM2.5", f"{pm.notna().sum():,} of {len(expected):,} ({present_pct:.1f}%)",
   present_pct >= MONTH_USABLE_PCT)

monthly = pm.notna().groupby(pm.index.to_period("M")).agg(["sum", "size"])
monthly.columns = ["hours_with_data", "hours_in_month"]
monthly["pct_complete"] = (monthly["hours_with_data"] / monthly["hours_in_month"] * 100).round(1)
monthly["usable"] = monthly["pct_complete"] >= MONTH_USABLE_PCT
n_bad = (~monthly["usable"]).sum()
dq("Completeness", "Table 1", f"months below {MONTH_USABLE_PCT}% complete", f"{n_bad} of {len(monthly)}", n_bad == 0)
monthly

In [ ]:
# Where are the gaps? Month × hour-of-day completeness. A pale column = the sensor tends to miss that hour of the day.
grid = (pm.notna().groupby([pm.index.to_period("M"), pm.index.hour]).mean().unstack() * 100)
fig, ax = plt.subplots(figsize=(11, 0.28 * len(grid) + 1.5))
im = ax.imshow(grid.values, aspect="auto", cmap=SEQ_CMAP, vmin=0, vmax=100)
ax.set_yticks(range(len(grid))); ax.set_yticklabels([str(p) for p in grid.index], fontsize=8)
ax.set_xticks(range(24)); ax.set_xticklabels(range(24), fontsize=8)
ax.set_xlabel("Hour of day (Philippine time)"); ax.grid(False)
ax.set_title(f"PM2.5 completeness by month and hour — {STATION_NAME}")
cb = fig.colorbar(im, ax=ax, fraction=0.025); cb.set_label("% of hours with a reading")
plt.tight_layout(); plt.show()

In [ ]:
# Gap lengths: how many gaps are short enough to fill, and how many hours are lost in long ones
gaps = true_runs(pm.isna())
bins   = [0, 1, 3, 6, 24, 24 * 7, np.inf]
labels = ["1 h", "2–3 h", "4–6 h", "7–24 h", "1–7 days", "> 7 days"]
gaps["length"] = pd.cut(gaps["hours"], bins=bins, labels=labels)
gap_summary = gaps.groupby("length", observed=False).agg(gaps=("hours", "size"), hours_missing=("hours", "sum"))
gap_summary["% of missing hours"] = (gap_summary["hours_missing"] / max(gaps["hours"].sum(), 1) * 100).round(1)
fillable = gaps.loc[gaps["hours"] <= MAX_INTERP_GAP_H, "hours"].sum()
dq("Completeness", "Table 1", f"missing hours fillable (gaps ≤ {MAX_INTERP_GAP_H} h)",
   f"{fillable:,} of {gaps['hours'].sum():,}", True)
print(f"Longest gap: {gaps['hours'].max() if len(gaps) else 0} hours")
display(gap_summary)
display(gaps.sort_values("hours", ascending=False).head(10))

In [ ]:
# Other tables and other columns
cov = aq_raw["percent_coverage"]
low_cov = (cov < MIN_HOUR_COVERAGE).sum()
dq("Completeness", "Table 1", f"hours built from < {MIN_HOUR_COVERAGE}% of expected readings", low_cov,
   low_cov / max(len(aq_raw), 1) < 0.05)
dq("Completeness", "Table 1", "missing percent_coverage values", cov.isna().sum(), True)

wx_missing = weather.drop(columns="datetime_local").isna().sum().sum()
dq("Completeness", "Table 2", "hours present", f"{len(weather):,} of {len(expected):,}", len(weather) == len(expected))
dq("Completeness", "Table 2", "missing weather values", wx_missing, wx_missing == 0)
dq("Completeness", "Table 3", "missing values", environment.isna().sum().sum(), environment.isna().sum().sum() == 0)
dq("Completeness", "Table 4", "hours present", f"{len(time_tbl):,} of {len(expected):,}", len(time_tbl) == len(expected))

**Decisions — completeness**
- **Gaps of ≤ 3 hours are interpolated; longer gaps stay missing** (Part 4). PM2.5 in Ortigas follows a traffic cycle with peaks that last a few hours. Drawing a straight line across 1–3 missing hours stays close to that cycle; across 4+ hours it would erase a rush-hour peak and invent values. The gap table above shows how much of the missing data this rule recovers. This follows the proposal: *"shorter gaps may be addressed through interpolation."*
- **Hourly averages built from < 75% of expected readings are treated as missing** (Part 3). 75% is the usual data-capture rule in air quality reporting: an hour averaged from too few readings may not represent the hour.
- **A month is "usable" at ≥ 75% completeness.** This defines the proposal's *"minimum quality threshold"* for the historical period. Months below it are kept in the table (their valid hours are still real data) but are listed in the report as low-confidence.
- **A 24-hour average needs ≥ 18 valid hours** (Part 4), the same 75% rule applied to the target.

📝 **Fill in after running:** overall completeness %, number of unusable months (and which), longest gap and its dates, and whether the heatmap shows the sensor missing specific hours of the day.

## 2.3 Validity
**Why it matters:** a value can be present but impossible (negative PM2.5, humidity of 120%). These rules define what an allowed value is. Ranges are wide on purpose: they catch errors, not unusual weather.

In [ ]:
rules = [
    # (table name, dataframe, column, min allowed, max allowed, reason)
    ("Table 1", aq_raw, "pm25", 0, PM25_MAX_VALID, "concentration can't be negative; >1000 µg/m³ is a sensor error"),
    ("Table 1", aq_raw, "percent_coverage", 0, 100, "a percentage"),
    ("Table 2", weather, "temp_c", 10, 45, "Metro Manila has never recorded below ~14 °C or above ~39 °C"),
    ("Table 2", weather, "humidity_pct", 0, 100, "a percentage"),
    ("Table 2", weather, "wind_speed_kmh", 0, 250, "250 km/h is above the strongest typhoon winds"),
    ("Table 2", weather, "wind_dir_deg", 0, 360, "compass degrees"),
    ("Table 2", weather, "precip_mm", 0, 200, "200 mm in one hour is beyond PH records"),
    ("Table 2", weather, "rain_mm", 0, 200, "same as precipitation"),
    ("Table 4", time_tbl, "hour", 0, 23, "hours of the day"),
    ("Table 4", time_tbl, "day_of_week", 0, 6, "Monday = 0 … Sunday = 6"),
]
validity = []
for tbl, df, col, lo, hi, why in rules:
    s = df[col].dropna()
    bad = ((s < lo) | (s > hi)).sum()
    validity.append({"table": tbl, "column": col, "allowed": f"{lo} – {hi}", "violations": bad,
                     "observed_min": s.min(), "observed_max": s.max(), "reason": why})
    dq("Validity", tbl, f"{col} within {lo}–{hi}", bad, bad == 0)

zeros = (aq_raw["pm25"] == 0).sum()
dq("Validity", "Table 1", "PM2.5 exactly 0 (possible but suspicious)", zeros, zeros / max(len(aq_raw), 1) < 0.01)
for col in ["is_weekend", "is_holiday"]:
    bad = (~time_tbl[col].isin([0, 1])).sum()
    dq("Validity", "Table 4", f"{col} is 0/1", bad, bad == 0)
env_counts = environment[["industrial_areas", "major_road_segments", "secondary_road_segments", "fuel_stations", "bus_stops"]]
dq("Validity", "Table 3", "counts are whole numbers ≥ 0", int((env_counts < 0).sum().sum()), (env_counts >= 0).all().all())
dq("Validity", "Table 1/2/4", "timestamps parsed as Asia/Manila datetimes",
   str(aq_raw["datetime_local"].dtype), all(str(d.dtype).endswith(f"{LOCAL_TZ}]") for d in
   [aq_raw["datetime_local"], weather["datetime_local"], time_tbl["datetime_local"]]))
pd.DataFrame(validity)

**Decision — invalid values:** PM2.5 values outside 0–1000 µg/m³ are set to missing in Part 3 (then treated like any other gap). They are not real air, so they can't be kept, and setting them to missing instead of deleting the row keeps the hourly calendar intact. A reading of exactly 0 is allowed but counted: some sensors report 0 when they fail.

📝 **Fill in after running:** number of violations per column (expected: none for weather and time tables).

## 2.4 Consistency
**Why it matters:** values must agree with each other. Inside a table (the average can't be above the maximum) and across tables (same time zone, same location, overlapping dates). Cross-table consistency decides whether the merge in Part 4 lines up the right hours.

In [ ]:
# --- Inside Table 1 ---
tol = 0.01
has_mm = aq_raw[["pm25", "pm25_min", "pm25_max"]].notna().all(axis=1)
bad = ((aq_raw.loc[has_mm, "pm25"] < aq_raw.loc[has_mm, "pm25_min"] - tol) |
       (aq_raw.loc[has_mm, "pm25"] > aq_raw.loc[has_mm, "pm25_max"] + tol)).sum()
dq("Consistency", "Table 1", "pm25_min ≤ pm25 ≤ pm25_max", bad, bad == 0)

units = stations.loc[stations["sensor_id"] == SENSOR_ID, "units"].unique()
dq("Consistency", "Table 1", "single unit of measurement", ", ".join(map(str, units)), len(units) == 1)

off_hour = (aq_raw["datetime_local"].dt.minute != 0).sum()
dq("Consistency", "Table 1", "timestamps on the hour (:00)", f"{off_hour} off-hour", off_hour == 0)

# --- Inside Table 2 ---
bad = (weather["rain_mm"] > weather["precip_mm"] + tol).sum()
dq("Consistency", "Table 2", "rain ≤ total precipitation", bad, bad == 0)
steps = weather["datetime_local"].diff().dropna().dt.total_seconds().div(3600)
dq("Consistency", "Table 2", "regular 1-hour steps", f"{(steps != 1).sum()} irregular", (steps != 1).sum() == 0)

# --- Across tables ---
tz_ok = len({str(d.dtype) for d in [aq_raw["datetime_local"], weather["datetime_local"], time_tbl["datetime_local"]]}) == 1
dq("Consistency", "Tables 1/2/4", "same time zone and datetime type", "yes" if tz_ok else "no", tz_ok)

for name, df in {"Table 1": aq_raw, "Table 2": weather}.items():
    starts_ok = df["datetime_local"].min() <= expected[0] + pd.Timedelta(days=7)
    ends_ok   = df["datetime_local"].max() >= expected[-1] - pd.Timedelta(days=7)
    dq("Consistency", name, "covers the study period",
       f"{df['datetime_local'].min():%Y-%m-%d} → {df['datetime_local'].max():%Y-%m-%d}", starts_ok and ends_ok)

meta = load_meta()
grid = meta.get("weather_grid")
if grid:
    km = haversine_km(STATION_LAT, STATION_LON, grid["lat"], grid["lon"])
    dq("Consistency", "Table 1 vs 2", "distance from station to weather grid point", f"{km:.1f} km", km <= 10)
env_ok = environment["station_name"].iloc[0] == STATION_NAME
dq("Consistency", "Table 1 vs 3", "environment counts centred on the same station", environment["station_name"].iloc[0], env_ok)

**Decisions — consistency**
- **All joins use `datetime_local` (Asia/Manila).** OpenAQ is delivered in UTC and Open-Meteo in local time; joining on raw UTC would shift weather by 8 hours against PM2.5. The time-zone check above confirms all three hourly tables now share one type.
- **Hours where PM2.5 is outside its own min–max are treated as invalid** (Part 3): the hourly average was calculated wrongly or the row is corrupted.
- **Timing offset is accepted and documented, not corrected.** PM2.5 at 13:00 = the 13:00–14:00 average; rain at 13:00 = the 12:00–13:00 total. Shifting rain by one hour would change nothing for a 24–72-hour forecast and would make the table harder to explain.
- **Weather is from a grid cell, not a weather station.** Open-Meteo's archive is a model reanalysis on a grid of roughly 10 km; the distance above shows how far its grid point sits from the station. One grid cell covers all of Mandaluyong, so this is fine for a single-station study, but it is a limitation to report.

📝 **Fill in after running:** any FLAGs here and the weather grid distance.

## 2.5 Accuracy
**Why it matters:** data can be complete, valid and consistent and still wrong (for example, a sensor stuck at one value). With no ground truth available, accuracy is judged with four indirect checks:
1. **Stuck sensor** — the same value for 6+ hours in a row. Real PM2.5 changes every hour.
2. **Agreement with the nearest other station** — two stations a few km apart should rise and fall together (daily means correlated).
3. **Sensor type** — reference stations (DENR-EMB) are more accurate than low-cost sensors, which read high in humid air.
4. **Weather vs known seasons** — the wet season (Jun–Nov) must be wetter than the dry season. If not, the weather data is misaligned.

In [ ]:
# 1. Stuck sensor (flatlines) — runs of the exact same PM2.5 value on the hourly calendar
same_run = pm.ne(pm.shift()).cumsum()
run_len = pm.groupby(same_run).transform("size").where(pm.notna())
flat_mask = run_len >= FLATLINE_HOURS
flat_runs = true_runs(flat_mask.fillna(False))
dq("Accuracy", "Table 1", f"stuck-sensor runs (same value ≥ {FLATLINE_HOURS} h)",
   f"{len(flat_runs)} runs, {int(flat_mask.sum())} hours", len(flat_runs) == 0)
if len(flat_runs):
    flat_runs["value"] = [pm.loc[s] for s in flat_runs["start"]]
    display(flat_runs.sort_values("hours", ascending=False).head(10))

In [ ]:
# 2. Agreement with the nearest other station (daily means, to smooth out hour-to-hour noise)
if neighbor is not None and len(neighbor):
    nb_info = meta.get("neighbor", {})
    nb_pm = (neighbor.drop_duplicates("datetime_utc").set_index("datetime_local")["pm25"]
               .where(lambda s: (s >= 0) & (s <= PM25_MAX_VALID)).reindex(expected))
    clean_pm = pm.where((pm >= 0) & (pm <= PM25_MAX_VALID))
    daily = pd.DataFrame({"ortigas": clean_pm.resample("D").mean(), "neighbor": nb_pm.resample("D").mean()}).dropna()
    if len(daily) >= 30:
        r = daily.corr().iloc[0, 1]
        bias = (daily["ortigas"] - daily["neighbor"]).mean()
        dq("Accuracy", "Table 1 vs 1b", f"daily correlation with {nb_info.get('name', 'neighbor')}",
           f"r = {r:.2f} over {len(daily)} days", r >= 0.7)
        print(f"Mean difference (Ortigas − neighbor): {bias:+.1f} µg/m³ | neighbor provider: {nb_info.get('provider')}, "
              f"{nb_info.get('distance_km')} km away")

        fig, ax = plt.subplots(figsize=(11, 3.2))
        ax.plot(daily.index, daily["ortigas"], color=BLUE, label=f"Ortigas ({STATION_NAME})")
        ax.plot(daily.index, daily["neighbor"], color=ORANGE, label=f"Neighbor ({nb_info.get('name', '')})")
        ax.set_ylabel("Daily mean PM2.5 (µg/m³)"); ax.legend(frameon=False, loc="upper left")
        ax.set_title("Accuracy cross-check: Ortigas vs nearest other station"); plt.tight_layout(); plt.show()
    else:
        dq("Accuracy", "Table 1 vs 1b", "neighbor cross-check", f"only {len(daily)} overlapping days", False)
else:
    dq("Accuracy", "Table 1 vs 1b", "neighbor cross-check", "no neighbor station", False)

In [ ]:
# 3. Sensor type
provider = str(station["provider"])
low_cost = any(k in provider.lower() for k in ["airgradient", "purpleair", "clarity", "sensor.community"])
dq("Accuracy", "Table 1", "data provider", provider + (" (low-cost sensor)" if low_cost else ""), not low_cost)

# 4. Weather vs known seasons
season = np.where(weather["datetime_local"].dt.month.isin([6, 7, 8, 9, 10, 11]), "wet", "dry")
rain_by_season = weather.groupby(season)["rain_mm"].sum() / weather.groupby(season)["datetime_local"].count() * 24
dq("Accuracy", "Table 2", "wet season rainier than dry season (mm/day)",
   f"wet {rain_by_season.get('wet', np.nan):.1f} vs dry {rain_by_season.get('dry', np.nan):.1f}",
   rain_by_season.get("wet", 0) > rain_by_season.get("dry", 0))
hot_month = weather.groupby(weather["datetime_local"].dt.month)["temp_c"].mean().idxmax()
dq("Accuracy", "Table 2", "hottest month is in Mar–May (Philippine summer)", f"month {hot_month}", hot_month in [3, 4, 5])

**Decisions — accuracy**
- **Stuck-sensor hours are set to missing** (Part 3). A sensor that repeats one value for 6+ hours is not measuring; keeping those hours would teach the model a flat line. 6 hours is long enough that real air almost never stays at the exact same value, short enough to catch a sensor that froze overnight.
- **The neighbor station is used only for checking, not for filling gaps.** Filling Ortigas gaps with another station's readings would mix two locations, which the proposal's scope rules out (single station, no spatial interpolation). A correlation of r ≥ 0.7 is taken as "the Ortigas station tracks real regional air."
- **If the provider is a low-cost sensor,** report it as a limitation: these sensors read high in humid air, which in Manila is most of the year.

📝 **Fill in after running:** number of stuck runs, the correlation r and mean difference with the neighbor, and the provider.

## 2.6 Timeliness
**Why it matters:** the model is meant to support current advisories. If the station stopped reporting months ago, or the latest months are mostly gaps, the training data no longer reflects current conditions.
Timeliness is measured three ways: how close the last reading is to the study end, whether the station is still reporting today, and how complete the most recent 3 months are.

In [ ]:
today = pd.Timestamp.now(tz=LOCAL_TZ)
last_pm = pm.last_valid_index()
lag_days = (expected[-1] - last_pm).days if last_pm is not None else np.nan
dq("Timeliness", "Table 1", "last PM2.5 reading vs study end", f"{last_pm:%Y-%m-%d %H:%M} ({lag_days} days before end)",
   lag_days <= 7)

if pd.notna(station.get("last_reading_utc")):
    last_seen = pd.to_datetime(station["last_reading_utc"], utc=True).tz_convert(LOCAL_TZ)
    days_silent = (today - last_seen).days
    dq("Timeliness", "Table 1a", "station's latest report (any date)", f"{last_seen:%Y-%m-%d} ({days_silent} days ago)",
       days_silent <= 7)

recent = pm[pm.index > expected[-1] - pd.Timedelta(days=90)]
dq("Timeliness", "Table 1", "completeness of the last 90 days", f"{recent.notna().mean() * 100:.1f}%",
   recent.notna().mean() * 100 >= MONTH_USABLE_PCT)

last_wx = weather.dropna(subset=["temp_c"])["datetime_local"].max()
dq("Timeliness", "Table 2", "last weather reading vs study end", f"{last_wx:%Y-%m-%d %H:%M}", last_wx >= expected[-1] - pd.Timedelta(days=1))

for key in ["table1_retrieved_utc", "table2_retrieved_utc", "table3_retrieved_utc"]:
    if key in meta:
        print(f"{key.replace('_retrieved_utc', ''):<8} retrieved on {meta[key]}")

**Decisions — timeliness**
- **The study window is fixed (`START_DATE`–`END_DATE`) and the retrieval date is recorded** in `run_metadata.json`. Re-running the APIs later can return slightly different numbers (stations re-upload data; Open-Meteo replaces preliminary weather with final values after a few weeks), so results are always tied to a saved snapshot. Groupmates should use `USE_SAVED_TABLES = True` so everyone analyses the same data.
- **If the last 90 days are below 75% complete,** the model's test period (the most recent 20%) will be thin; report it as a limitation, and consider moving `END_DATE` back to the last well-covered month.

📝 **Fill in after running:** last reading date, whether the station is still reporting, and the last-90-day completeness.

## 2.7 Data-quality scorecard
Every check above in one table. `FLAG` doesn't always mean the data is unusable; it means the result needs a decision or a note in the report.

In [ ]:
scorecard = pd.DataFrame(dq_rows)
summary = scorecard.groupby("dimension", sort=False)["status"].agg(
    checks="size", flags=lambda s: (s == "FLAG").sum()).reset_index()
display(summary)
scorecard.to_csv(f"{DATA_DIR}/data_quality_scorecard.csv", index=False)
scorecard

---
# Part 3 — Outliers

**Group decision: keep all real spikes; remove only values that can't be real air.**

The notebook separates two kinds of unusual values:

| Kind | Examples | What happens | Why |
|---|---|---|---|
| **Errors** (not real air) | negative or > 1000 µg/m³, average outside its own min–max, stuck sensor, hour built from < 75% of readings, duplicate hours | set to **missing** (row kept) | they describe the sensor, not the air; treated like any other gap |
| **Real extremes** | New Year's fireworks, transboundary haze, typhoon rain, a short traffic spike | **kept** and **labelled** with flag columns | they are real exposure events; removing them would hide the most dangerous hours |

Outliers are *labelled*, not removed, so the group can later report results with and without them (for example, model accuracy excluding the haze period), without re-running the cleaning.

## 3.1 Remove errors (Table 1)

In [ ]:
# 1. Duplicates: keep the row with the most raw readings; on a tie keep the later upload (see 2.1)
aq = (aq_raw.reset_index(names="api_order")
        .sort_values(["datetime_utc", "observed_count", "api_order"], na_position="first", kind="stable")
        .drop_duplicates("datetime_utc", keep="last"))

# 2. Put on the full hourly calendar
aq_clean = (aq.set_index("datetime_local")[["pm25", "pm25_min", "pm25_max", "percent_coverage"]]
              .reindex(expected))
aq_clean.index.name = "datetime_local"
aq_clean["pm25_raw"] = aq_clean["pm25"]
aq_clean["invalid_reason"] = pd.Series(pd.NA, index=aq_clean.index, dtype="string")

def mark_invalid(mask, reason):
    mask = mask.fillna(False) & aq_clean["pm25"].notna()
    aq_clean.loc[mask, "invalid_reason"] = reason
    aq_clean.loc[mask, "pm25"] = np.nan
    print(f"{reason:<22} {int(mask.sum()):>6} hours set to missing")

# 3. Rule-based errors (order matters only for which reason is recorded)
mark_invalid((aq_clean["pm25"] < 0) | (aq_clean["pm25"] > PM25_MAX_VALID), "out_of_range")
mark_invalid((aq_clean["pm25"] < aq_clean["pm25_min"] - 0.01) | (aq_clean["pm25"] > aq_clean["pm25_max"] + 0.01), "outside_own_min_max")
mark_invalid(aq_clean["percent_coverage"] < MIN_HOUR_COVERAGE, "low_coverage")

# 4. Stuck sensor: same value for FLATLINE_HOURS+ consecutive hours
s = aq_clean["pm25"]
run_len = s.groupby(s.ne(s.shift()).cumsum()).transform("size").where(s.notna())
mark_invalid(run_len >= FLATLINE_HOURS, "stuck_sensor")

print(f"\nHours with valid PM2.5: {aq_clean['pm25'].notna().sum():,} of {len(aq_clean):,} "
      f"({aq_clean['pm25'].notna().mean() * 100:.1f}%)")
aq_clean["invalid_reason"].value_counts(dropna=True)

## 3.2 Find and label real extremes (Table 1)
Three labels, each answering a different question:
- **`is_extreme_iqr`** — is the hour unusually high *for the whole study period*? Rule: above Q3 + 3 × IQR. The 3 × IQR fence (instead of the usual 1.5) is used because PM2.5 is right-skewed: 1.5 × IQR would flag ordinary bad-traffic days.
- **`is_isolated_spike`** — is it a one-hour jump that drops straight back? Rule: robust z-score > 5 against the surrounding 24 hours while both neighbouring hours are normal (z < 3). These can be a passing truck *or* a sensor glitch; we can't tell, so they are kept.
- **`event`** — does it fall inside a known event window (New Year's fireworks, the September 2026 haze)? Edit `KNOWN_EVENTS` if the group finds exact dates.

In [ ]:
s = aq_clean["pm25"]

# Study-wide extreme values
q1, q3 = s.quantile([0.25, 0.75])
upper_fence = q3 + 3 * (q3 - q1)
aq_clean["is_extreme_iqr"] = (s > upper_fence).astype(int)

# Isolated one-hour spikes (robust z-score against a centred 24-h window)
med = s.rolling(24, center=True, min_periods=12).median()
mad = (s - med).abs().rolling(24, center=True, min_periods=12).median()
robust_z = (s - med) / (1.4826 * mad.replace(0, np.nan))
aq_clean["is_isolated_spike"] = ((robust_z > 5) & (robust_z.shift(1) < 3) & (robust_z.shift(-1) < 3)).astype(int)

# Known events (labels only; never used as model features, because they can't be known ahead of time)
KNOWN_EVENTS = {f"new_year_{y}": (f"{y-1}-12-31 18:00", f"{y}-01-01 12:00")
                for y in range(int(START_DATE[:4]) + 1, int(END_DATE[:4]) + 1)}
KNOWN_EVENTS["haze_sep_2026"] = ("2026-09-01 00:00", "2026-09-30 23:00")   # narrow to the dates in the news reports if known
aq_clean["event"] = pd.Series(pd.NA, index=aq_clean.index, dtype="string")
for name, (a, b) in KNOWN_EVENTS.items():
    aq_clean.loc[pd.Timestamp(a, tz=LOCAL_TZ):pd.Timestamp(b, tz=LOCAL_TZ), "event"] = name

print(f"Q1 = {q1:.1f}, Q3 = {q3:.1f}, upper fence (Q3 + 3×IQR) = {upper_fence:.1f} µg/m³")
outlier_summary = pd.DataFrame({
    "hours": [aq_clean["is_extreme_iqr"].sum(), aq_clean["is_isolated_spike"].sum()],
    "inside a known event": [((aq_clean["is_extreme_iqr"] == 1) & aq_clean["event"].notna()).sum(),
                             ((aq_clean["is_isolated_spike"] == 1) & aq_clean["event"].notna()).sum()],
}, index=["extreme (Q3 + 3×IQR)", "isolated 1-hour spike"])
outlier_summary["% of valid hours"] = (outlier_summary["hours"] / s.notna().sum() * 100).round(2)
outlier_summary

In [ ]:
# The 15 highest hours — check whether each one has an explanation (event, holiday, haze)
top = aq_clean.nlargest(15, "pm25")[["pm25", "is_extreme_iqr", "is_isolated_spike", "event"]].copy()
top["day"] = top.index.day_name()
top

In [ ]:
# Daily PM2.5 with extreme hours and DENR thresholds
daily_mean = s.resample("D").mean()
ext = aq_clean[aq_clean["is_extreme_iqr"] == 1]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(daily_mean.index, daily_mean, color=BLUE, linewidth=1.2, label="Daily mean PM2.5")
ax.scatter(ext.index, ext["pm25"], s=10, color=ORANGE, zorder=3, label=f"Extreme hour (> {upper_fence:.0f} µg/m³)")
for name, (a, b) in KNOWN_EVENTS.items():
    ax.axvspan(pd.Timestamp(a, tz=LOCAL_TZ), pd.Timestamp(b, tz=LOCAL_TZ), color=GRAY, alpha=0.15, linewidth=0)
for edge, lab in zip(DENR_BINS[1:-1], DENR_LABELS[1:]):   # DENR thresholds, labelled in the right margin
    ax.axhline(edge, color=GRAY, linewidth=0.6, linestyle="--")
    ax.annotate(f"{lab} ({edge}+)", xy=(1.005, edge), xycoords=("axes fraction", "data"),
                fontsize=6.5, color=INK, va="center", annotation_clip=False)
ax.set_ylabel("PM2.5 (µg/m³)"); ax.legend(frameon=False, loc="upper right", fontsize=8)
ax.set_title("PM2.5 over the study period — grey bands = known events")
plt.tight_layout(); plt.show()

In [ ]:
# Monthly spread: do extremes cluster in particular months?
by_month = [g.dropna().values for _, g in s.groupby(s.index.to_period("M"))]
labels_m = [str(p) for p in s.groupby(s.index.to_period("M")).size().index]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.boxplot(by_month, whis=3, widths=0.6, showfliers=True,
           flierprops=dict(marker="o", markersize=2, markerfacecolor=ORANGE, markeredgecolor="none"),
           medianprops=dict(color=BLUE, linewidth=1.5), boxprops=dict(color=GRAY), whiskerprops=dict(color=GRAY),
           capprops=dict(color=GRAY))
ax.set_xticks(range(1, len(labels_m) + 1)); ax.set_xticklabels(labels_m, rotation=60, fontsize=7)
ax.set_ylabel("Hourly PM2.5 (µg/m³)"); ax.set_title("Hourly PM2.5 by month (whiskers at 3 × IQR)")
plt.tight_layout(); plt.show()

## 3.3 Weather outliers (Table 2)
Weather values already passed the validity rules, so anything unusual left is real weather. They are labelled and kept:
- **Temperature and humidity:** robust z-score > 4 against the whole period.
- **Rain:** not tested with IQR. Most hours have 0 mm, so every rainy hour would count as an outlier. Instead the heaviest rain hours are listed to confirm they fall in the wet season / known typhoons.

In [ ]:
def robust_z_all(x):
    m = x.median(); d = (x - m).abs().median()
    return (x - m) / (1.4826 * d) if d else x * 0

weather_flags = pd.DataFrame({"datetime_local": weather["datetime_local"]})
for col in ["temp_c", "humidity_pct", "wind_speed_kmh"]:
    weather_flags[f"{col}_outlier"] = (robust_z_all(weather[col]).abs() > 4).astype(int)
    print(f"{col:<16} {weather_flags[f'{col}_outlier'].sum():>5} unusual hours (kept)")

print("\nHeaviest rain hours (kept):")
heavy = weather.nlargest(10, "rain_mm")[["datetime_local", "rain_mm", "wind_speed_kmh"]]
heavy["season"] = np.where(heavy["datetime_local"].dt.month.isin([6, 7, 8, 9, 10, 11]), "wet", "dry")
heavy

## 3.4 Justification — why real spikes are kept

1. **They are the hours the project exists for.** The goal is early warning of unhealthy air. The *Acutely Unhealthy* and *Emergency* classes come almost entirely from extreme hours; removing them would leave the model with no examples of the most important classes, and it would learn to under-predict risk. For a health advisory, under-predicting is the costlier mistake.
2. **Removing them would bias the data.** Dropping high values lowers the averages and makes Ortigas look cleaner than it is, which would make the 24-hour target (and every lag feature) wrong for days around each event.
3. **The models can handle them.** A Decision Tree splits on thresholds, so one extreme value doesn't distort it. Logistic Regression is more sensitive; for it, PM2.5 features can be log-transformed (`np.log1p`) and scaled in the modeling step, which reduces the pull of extremes without deleting them.
4. **It stays consistent with the proposal's delimitations.** The proposal says the model is *not expected* to predict haze or fireworks. Keeping the data does not change that: the `event` label lets the evaluation report accuracy with and without those periods, so missed haze hours are shown as a known blind spot rather than a model failure.
5. **Errors are still removed.** Only values that can't be real air (impossible ranges, internal contradictions, stuck sensor, too few readings) are set to missing, and every one is recorded in `invalid_reason`, so the count of removed values can be reported exactly.

**Isolated spikes** are the one grey area: a one-hour spike can be a truck idling beside the station or a sensor glitch. They are kept and labelled. Setting `is_isolated_spike` hours to missing later is a one-line change if the group decides otherwise.

📝 **Fill in after running:** how many hours were set to missing (by reason), how many extreme hours were found, and whether the top-15 hours line up with known events.

---
# Part 4 — Merging all four tables

**Plan**

```
Table 4 (time, every hour)  ── spine ──┐
Table 1 (PM2.5, cleaned)    ── LEFT JOIN on datetime_local ──┤
Table 2 (weather)           ── LEFT JOIN on datetime_local ──┤──► merged_hourly (one row per hour)
Table 3 (environment, 1 row)── CROSS JOIN (copied to every row) ─┘
```

| Step | Decision | Justification |
|---|---|---|
| Spine | Start from **Table 4** | It is the only table with every hour of the study period. Joining onto it keeps missing PM2.5 hours as visible blanks; starting from Table 1 (or an inner join) would silently delete them, and lag/rolling features would then compare hours that aren't 1 or 24 hours apart. |
| Join key | **`datetime_local`** (Asia/Manila) | The one column shared by Tables 1, 2 and 4, already checked for a common time zone in 2.4. |
| Join type | **Left** joins, checked as **one-to-one** | Every spine hour stays exactly once; pandas stops with an error if any table has duplicate hours (guarding the Uniqueness result). |
| Table 3 | **Cross join** (copied to all rows) | It has no time column, and with one station there is nothing else to match on. As agreed, it is included; Part 4.4 shows the consequence. |
| Columns dropped | `pm25_min`, `pm25_max`, `percent_coverage`, `precip_mm`, `day_name`, station coordinates | Used for quality checks only, or duplicate another column (see 4.1). |

## 4.1 Select columns and join

In [ ]:
aq_part  = aq_clean[["pm25", "invalid_reason", "is_extreme_iqr", "is_isolated_spike", "event"]].reset_index()
aq_part["datetime_local"] = aq_part["datetime_local"].astype(f"datetime64[ns, {LOCAL_TZ}]")

r_rain = weather[["rain_mm", "precip_mm"]].corr().iloc[0, 1]
print(f"Correlation rain_mm vs precip_mm = {r_rain:.3f} → keep rain_mm only (precip_mm adds almost no new information)")
wx_part  = weather[["datetime_local", "temp_c", "humidity_pct", "wind_speed_kmh", "wind_dir_deg", "rain_mm"]]
env_part = environment.drop(columns=["lat", "lon", "radius_m"])
time_part = time_tbl.drop(columns=["day_name"])

merged = (time_part
          .merge(aq_part, on="datetime_local", how="left", validate="one_to_one")
          .merge(wx_part, on="datetime_local", how="left", validate="one_to_one")
          .merge(env_part, how="cross"))

assert len(merged) == len(time_tbl), "Row count changed during the merge — check for duplicate hours"
assert merged["datetime_local"].is_monotonic_increasing and merged["datetime_local"].is_unique
print(f"Merged table: {merged.shape[0]:,} rows × {merged.shape[1]} columns")
print(f"PM2.5 missing after merge: {merged['pm25'].isna().sum():,} hours ({merged['pm25'].isna().mean() * 100:.1f}%)")
merged.head()

## 4.2 Fill short gaps
Gaps of ≤ `MAX_INTERP_GAP_H` hours are filled by time-based linear interpolation; longer gaps stay missing (reason in 2.2). Only gaps with a valid reading on **both** sides are filled, so nothing is extrapolated past the start or end of the data.
Every filled hour is marked in `pm25_interpolated`, so it can be reported or excluded later.

In [ ]:
def interpolate_short_gaps(s, max_gap):
    """Interpolate only runs of missing values that are max_gap hours long or shorter."""
    miss = s.isna()
    run_len = miss.groupby(miss.ne(miss.shift()).cumsum()).transform("size")
    fillable = miss & (run_len <= max_gap)
    interp = s.interpolate(method="time", limit_area="inside")
    filled = s.where(~fillable, interp)
    return filled, (fillable & filled.notna()).astype(int)

merged = merged.set_index("datetime_local")
before = merged["pm25"].isna().sum()
merged["pm25"], merged["pm25_interpolated"] = interpolate_short_gaps(merged["pm25"], MAX_INTERP_GAP_H)
for col in ["temp_c", "humidity_pct", "wind_speed_kmh", "wind_dir_deg", "rain_mm"]:
    merged[col], _ = interpolate_short_gaps(merged[col], MAX_INTERP_GAP_H)
after = merged["pm25"].isna().sum()
print(f"PM2.5 missing hours: {before:,} → {after:,}  ({before - after:,} filled by interpolation)")

## 4.3 Build the target and forecasting features

**Target — DENR risk category of the 24-hour average.** DENR's PM2.5 breakpoints are defined for 24-hour averages, so the label is computed from `pm25_24h`, the average of the current and previous 23 hours (needs ≥ 18 valid hours). This keeps the proposal's DENR labels while staying hourly, so time-of-day features stay meaningful (group decision).

**Forecast targets.** The category 24, 48 and 72 hours ahead (`target_24h`, `target_48h`, `target_72h`) — covering both the 48-hour window in the Scope and the 72 hours in the Objectives.

**Features known at forecast time.** Only past or current values are used as inputs (current PM2.5, PM2.5 1 h and 24 h ago, the current 24-h average, last-24-h rain and wind). Using anything from the future would leak the answer into the inputs. `pm25_24h` at hour *t* covers *t-23…t*, while `target_24h` covers *t+1…t+24*, so the two windows never overlap.

Wind direction is stored in degrees, where 359° and 1° are almost the same; it is converted to sine/cosine so the model sees them as close.

In [ ]:
pm25 = merged["pm25"]
merged["pm25_24h"]     = pm25.rolling(24, min_periods=ROLLING_MIN_HOURS).mean()
merged["aqi_category"] = pd.cut(merged["pm25_24h"], bins=DENR_BINS, labels=DENR_LABELS, right=True)

# Forecast targets (category h hours ahead)
for h in [24, 48, 72]:
    merged[f"target_{h}h"] = merged["aqi_category"].shift(-h)

# Features known at the forecast hour
merged["pm25_lag1"]   = pm25.shift(1)
merged["pm25_lag24"]  = pm25.shift(24)
merged["rain_24h_mm"] = merged["rain_mm"].rolling(24, min_periods=ROLLING_MIN_HOURS).sum()
merged["wind_24h_kmh"] = merged["wind_speed_kmh"].rolling(24, min_periods=ROLLING_MIN_HOURS).mean()
merged["wind_dir_sin"] = np.sin(np.radians(merged["wind_dir_deg"]))
merged["wind_dir_cos"] = np.cos(np.radians(merged["wind_dir_deg"]))

merged = merged.reset_index()
merged["aqi_category"].value_counts(dropna=False).reindex(DENR_LABELS + [np.nan])

## 4.4 Feature list, constant columns and the model-ready table

In [ ]:
FEATURE_COLS = [
    # PM2.5 history
    "pm25", "pm25_lag1", "pm25_lag24", "pm25_24h",
    # weather
    "temp_c", "humidity_pct", "wind_speed_kmh", "wind_dir_sin", "wind_dir_cos", "rain_mm", "rain_24h_mm", "wind_24h_kmh",
    # time of day / calendar
    "hour", "day_of_week", "is_weekend", "is_holiday", "month", "day_period", "season",
    # urban environment (Table 3)
    "industrial_areas", "major_road_segments", "secondary_road_segments", "fuel_stations", "bus_stops",
]
TARGET_COLS = ["target_24h", "target_48h", "target_72h"]
LABEL_COLS  = ["invalid_reason", "is_extreme_iqr", "is_isolated_spike", "event", "pm25_interpolated"]   # for reporting, not features

constant = [c for c in FEATURE_COLS if merged[c].nunique(dropna=True) <= 1]
print("Constant feature columns (same value on every row):", constant)

⚠️ **Environment columns are constant.** With one station, every row has the same road and industry counts. A model can't learn from a column that never changes: a Decision Tree will never split on it and Logistic Regression will give it a coefficient of ~0, so it will show **zero importance**. They are kept in the merged table as agreed, but the report should say this directly: *with a single monitoring point, urban environmental factors describe the site but can't be tested as predictors.* They would only become useful with several stations in different surroundings.

In [ ]:
# Rows usable for each forecast horizon (all model features present, target present)
model_features = [c for c in FEATURE_COLS if c not in constant]
usable = pd.DataFrame({h: [merged[model_features + [h]].notna().all(axis=1).sum()] for h in TARGET_COLS},
                      index=["usable rows"]).T
usable["% of all hours"] = (usable["usable rows"] / len(merged) * 100).round(1)
display(usable)

model_ready = merged.dropna(subset=model_features + TARGET_COLS).reset_index(drop=True)
print(f"Model-ready rows (all 3 horizons present): {len(model_ready):,} of {len(merged):,}")
print(f"Share of model-ready rows whose PM2.5 was interpolated: {model_ready['pm25_interpolated'].mean() * 100:.1f}%")

# Time-based 80/20 split preview — train on the past, test on the most recent 20%
cutoff = model_ready["datetime_local"].iloc[int(len(model_ready) * 0.8)]
print(f"80/20 split by time: train before {cutoff:%Y-%m-%d %H:%M}, test from then on")

merged.to_csv(f"{DATA_DIR}/merged_hourly_full.csv", index=False)
model_ready.to_csv(f"{DATA_DIR}/model_ready_hourly.csv", index=False)
print("Saved merged_hourly_full.csv and model_ready_hourly.csv to", DATA_DIR)

**Why split by time, not randomly (for the modeling notebook):** neighbouring hours are nearly identical. A random 80/20 split puts 13:00 in training and 14:00 in testing, so the model is tested on hours it has effectively already seen and accuracy looks better than it will be on real future days. Training on the earlier 80% of the timeline and testing on the last 20% matches how the model would actually be used. This keeps the proposal's 80/20 ratio and only changes *how* the rows are split.

📝 **Fill in after running:** rows lost to long gaps, model-ready rows per horizon, share interpolated, and the split date.

---
# Part 5 — EDA on the merged table

In [ ]:
# Class balance of the hourly DENR category
counts = merged["aqi_category"].value_counts().reindex(DENR_LABELS, fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3.6))
bars = ax.bar(DENR_LABELS, counts.values, color=DENR_COLORS, width=0.65, edgecolor="white", linewidth=2)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / max(counts.sum(), 1) * 100:.1f}%)",
            ha="center", va="bottom", fontsize=8, color=INK)
ax.set_ylabel("Hours"); ax.set_title("DENR risk category of the 24-h PM2.5 average (all hours)")
ax.margins(y=0.2); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**What to look for:** most hours will be *Good* or *Fair*; the top categories may have very few hours or none. A model can score high accuracy by always predicting *Good*, so the modeling step should also report per-class recall and the confusion matrix (already planned in the proposal), and may need `class_weight="balanced"`. Categories with no hours at all can't be predicted and should be named in the limitations.

In [ ]:
# Daily pattern: average PM2.5 by hour of day, weekdays vs weekends (rush-hour effect)
prof = merged.groupby(["is_weekend", "hour"])["pm25"].mean().unstack(0)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(prof.index, prof[0], color=BLUE, marker="o", markersize=4, label="Weekday")
ax.plot(prof.index, prof[1], color=ORANGE, marker="o", markersize=4, label="Weekend")
ax.margins(y=0.15)
for band, (a, b) in {"morning rush": (6, 9), "evening rush": (17, 20)}.items():
    ax.axvspan(a - 0.5, b + 0.5, color=GRAY, alpha=0.12, linewidth=0)
    ax.text((a + b) / 2, ax.get_ylim()[1], band, ha="center", va="top", fontsize=8, color=INK)
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("Hour of day"); ax.set_ylabel("Mean PM2.5 (µg/m³)")
ax.legend(frameon=False); ax.set_title("Average PM2.5 by hour — weekday vs weekend")
plt.tight_layout(); plt.show()

In [ ]:
# How each numeric feature moves with the 24-h PM2.5 average (Spearman: robust to the kept extremes)
num_cols = [c for c in model_features if merged[c].dtype.kind in "if"] + ["pm25_24h"]
num_cols = list(dict.fromkeys(num_cols))
corr = merged[num_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr.values, cmap=DIV_CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols, fontsize=8); ax.grid(False)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(corr.values[i, j]) > 0.6 else INK)
fig.colorbar(im, ax=ax, fraction=0.04).set_label("Spearman correlation")
ax.set_title("Correlation between features"); plt.tight_layout(); plt.show()

corr["pm25_24h"].drop("pm25_24h").sort_values().to_frame("correlation with pm25_24h")

In [ ]:
# PM2.5 by time-of-day period and season
order = ["morning_rush", "midday", "evening_rush", "night"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (season_name, g) in zip(axes, merged.groupby("season")):
    data = [g.loc[g["day_period"] == p, "pm25"].dropna() for p in order]
    ax.boxplot(data, whis=3, showfliers=False, widths=0.6,
               medianprops=dict(color=BLUE, linewidth=1.5), boxprops=dict(color=GRAY),
               whiskerprops=dict(color=GRAY), capprops=dict(color=GRAY))
    ax.set_xticks(range(1, 5)); ax.set_xticklabels([p.replace("_", " ") for p in order], fontsize=8)
    ax.set_title(f"{season_name.capitalize()} season")
axes[0].set_ylabel("Hourly PM2.5 (µg/m³)")
plt.suptitle("PM2.5 by time of day (outlier dots hidden for readability; they are still in the data)", fontsize=10)
plt.tight_layout(); plt.show()

📝 **Fill in after running:**
- Class balance: which categories have few or no hours?
- Daily pattern: do the rush-hour bands show peaks? Is the weekend lower?
- Correlations: which weather variables move with PM2.5 (expected: rain and wind negative)? Are any two features so correlated (> 0.9) that one should be dropped for Logistic Regression?
- Season: is the dry season higher?

---
# Part 6 — Decision summary

| # | Area | Decision | Justification |
|---|---|---|---|
| 1 | Scope | One station (Ortigas), hourly data, `START_DATE`–`END_DATE` | Group scope; one well-covered station gives a cleaner study than several patchy ones |
| 2 | Uniqueness | Duplicate hours → keep the row with the most raw readings (tie: latest upload) | More readings = more reliable average; latest upload usually corrects earlier ones |
| 3 | Validity | PM2.5 < 0 or > 1000 µg/m³ → missing | Physically impossible / sensor error |
| 4 | Consistency | Average outside its own min–max → missing | The row contradicts itself |
| 5 | Completeness | Hours built from < 75% of expected readings → missing | Usual 75% data-capture rule; too few readings may not represent the hour |
| 6 | Accuracy | Same value 6+ hours in a row → missing | Stuck sensor; real PM2.5 changes every hour |
| 7 | Accuracy | Nearest other station used only to check, never to fill | Filling would mix locations, which the scope rules out |
| 8 | Outliers | Real extremes kept and labelled (`is_extreme_iqr`, `is_isolated_spike`, `event`) | Needed for the high-risk classes; removing them biases the model toward under-warning; labels allow results with/without events |
| 9 | Outliers | Extreme fence at Q3 + 3 × IQR | PM2.5 is right-skewed; 1.5 × IQR would flag ordinary bad days |
| 10 | Merge | Table 4 as spine, left joins on `datetime_local`, one-to-one checked | Keeps every hour; gaps stay visible; lags stay correctly spaced |
| 11 | Merge | Table 3 cross-joined (copied to every row) | Group decision; documented as constant, zero predictive value with one station |
| 12 | Merge | `precip_mm`, `pm25_min/max`, `percent_coverage`, `day_name` dropped | Duplicate another column or used only for quality checks |
| 13 | Gaps | Interpolate gaps ≤ 3 h only, both sides needed; flag filled hours | Short gaps follow the daily cycle; longer ones would erase rush-hour peaks |
| 14 | Timing | 1-hour offset between PM2.5 (hour after) and rain (hour before) left as is | Negligible for 24–72 h forecasts; documented |
| 15 | Target | DENR category of the trailing 24-h average (≥ 18 valid hours), hourly rows | DENR breakpoints are for 24-h averages; hourly rows keep time-of-day features meaningful |
| 16 | Target | Targets at +24 h, +48 h, +72 h | Covers both the 48 h (Scope) and 72 h (Objectives) in the proposal |
| 17 | Features | Only past/current values as inputs; event labels never used as features | Prevents leakage; events can't be known ahead |
| 18 | Features | Wind direction → sine/cosine | 359° and 1° are neighbours |
| 19 | Split (next notebook) | 80/20 by time, not random | Random split leaks neighbouring hours and inflates accuracy |
| 20 | Timeliness | Data saved with retrieval date; groupmates load saved tables | APIs revise data; everyone must analyse the same snapshot |

Add any new decision to this table **and** to the group's Decision Log (see the documentation guide).